### denoising diffusion probabilistic model

## Data

In [ ]:
import io, re, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df=pd.read_parquet("../../datos/faithful_1.21.1_32x32.parquet")
print(device)

img_size=32
embed_dim=128

## Code

In [ ]:
def mostrar_resultados(resultados):
    """
    Muestra una lista de imágenes generadas con sus etiquetas.
    """
    plt.figure(figsize=(8, 2))
    for i, (img, label) in enumerate(resultados):
        plt.subplot(1, len(resultados), i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(label)
    plt.tight_layout()
    plt.show()

def mostrar_metricas(epocas, valores, num_epochs, label:str=""):
    plt.figure(figsize=(6, 2))
    plt.xlim(0, num_epochs)
    # Ajuste dinámico del límite Y basado en el histórico
    max_val = max(valores)
    plt.ylim(0, max_val * 1.1)
    
    plt.xlabel('Epoca')
    plt.ylabel(label)
    plt.plot(epocas, valores, color="r", marker='o', linewidth=1, label=label)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def generar_imagenes(modelo, txt_emb, prompt, n_samples=4, device=device, return_pil = True):
    """Genera n_samples del promt si es único y si es lista de promts, genera 1 por cada promt"""

    if isinstance(prompt, str):
        lista_prompts = [prompt] * n_samples
    else:
        lista_prompts = prompt
        n_samples = len(lista_prompts)
    
    modelo.eval()
    txt_emb.eval()
    with torch.no_grad():
        # Embedding condicional del texto
        cond_emb = txt_emb(lista_prompts).to(device)

        # Ruido aleatorio
        x = torch.randn(n_samples, 3, img_size, img_size, device=device)

        #  Parámetros del modelo (difusión)
        betas = modelo.betas.to(device)
        alphas = modelo.alphas.to(device)
        alphas_cumprod = modelo.alphas_cumprod.to(device)
        timesteps = modelo.timesteps

        # generar imagenes
        for t in reversed(range(timesteps)):
            t_tensor = torch.full((n_samples,), t, device=device, dtype=torch.long)
            noise_pred = modelo(x, t_tensor, cond_emb)

            beta_t = betas[t]
            alpha_t = alphas[t]
            alpha_bar_t = alphas_cumprod[t]

            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = 0

            x = (
                (1 / torch.sqrt(alpha_t))
                * (x - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * noise_pred)
                + torch.sqrt(beta_t) * noise
            )
        # desnormalizar
        x = (x * 0.5 + 0.5).clamp(0, 1)

        # Convertir a PIL y devolver
        if return_pil: 
            return [(TF.to_pil_image(img.cpu()), p) for img, p in zip(x, lista_prompts)] 
        # Devolver tensores pytorch
        return x

# Dataset único (con augment)
class MinecraftDataset(Dataset):
    def __init__(self, rows: List[dict], image_size=img_size, augment=False):
        self.rows = rows
        self.image_size = image_size
        self.augment = augment
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        img = Image.open(io.BytesIO(r["image"]["bytes"])).convert("RGB")
        img_t = self.transform(img)
        if self.augment and torch.rand(1).item() > 0.5:
            img_t = torch.flip(img_t, dims=[2])
        label = r["label"]
        return img_t, label


# Embedding textual (por palabra)
class TextEmbedding(nn.Module):
    def __init__(self, vocab, embed_dim=embed_dim):
        super().__init__()
        self.vocab = vocab
        self.stoi = {w: i for i, w in enumerate(vocab)}
        self.embedding = nn.Embedding(len(vocab), embed_dim)

    @staticmethod
    def build_vocab(labels):
        words = set()
        words.add("unknown")
        for lbl in labels:
            for w in re.findall(r"\w+", lbl.lower()):
                words.add(w)
        return sorted(words)

    def forward(self, label_texts: List[str]):
        device = next(self.embedding.parameters()).device
        vectors = []
        for text in label_texts:
            tokens = [w for w in re.findall(r"\w+", text.lower()) if w in self.stoi]
            if not tokens:
                tokens = ["unknown"]
            idxs = torch.tensor([self.stoi[w] for w in tokens], device=device)
            emb = self.embedding(idxs).mean(dim=0)
            vectors.append(emb)
        return torch.stack(vectors)

#### Crear Modelo

In [ ]:
# Bloques básicos
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, embed_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_fc = nn.Linear(1, out_ch)
        self.cond_fc = nn.Linear(embed_dim, out_ch)
        self.norm = nn.BatchNorm2d(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t, cond):
        h = F.silu(self.conv1(x))
        t_emb = self.time_fc(t.unsqueeze(-1)).view(t.size(0), -1, 1, 1)
        c_emb = self.cond_fc(cond).view(cond.size(0), -1, 1, 1)
        h = h + t_emb + c_emb
        h = F.silu(self.norm(self.conv2(h)))
        return h + self.skip(x)


class MiniUNet(nn.Module): 
    def __init__(self, embed_dim=embed_dim, base_ch=64, timesteps=100):
        super().__init__()
        self.timesteps = timesteps
        # Parámetros de difusión
        self.register_buffer("betas", torch.linspace(1e-4, 0.02, timesteps))
        self.register_buffer("alphas", 1.0 - self.betas)
        self.register_buffer("alphas_cumprod", torch.cumprod(self.alphas, dim=0))

        # Encoder
        self.down1 = ResidualBlock(3, base_ch, embed_dim)
        self.down2 = ResidualBlock(base_ch, base_ch * 2, embed_dim)
        self.down3 = ResidualBlock(base_ch * 2, base_ch * 4, embed_dim)
        self.pool = nn.AvgPool2d(2)

        # Middle
        self.mid1 = ResidualBlock(base_ch * 4, base_ch * 4, embed_dim)
        self.mid2 = ResidualBlock(base_ch * 4, base_ch * 4, embed_dim)

        # Decoder
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = ResidualBlock(base_ch * 4, base_ch * 2, embed_dim)
        self.up2 = ResidualBlock(base_ch * 2, base_ch, embed_dim)
        self.out = nn.Conv2d(base_ch, 3, 1)

    def forward(self, x, t, cond):
        # Normalizar t en [0,1]
        t = t.float() / self.timesteps

        h1 = self.down1(x, t, cond)
        h2 = self.down2(self.pool(h1), t, cond)
        h3 = self.down3(self.pool(h2), t, cond)

        h_mid1 = self.mid1(h3, t, cond)
        h_mid2 = self.mid2(h_mid1, t, cond)

        h = self.up(h_mid2)
        h = self.up3(h, t, cond)
        h = self.up(h)
        h = self.up2(h, t, cond)
        out = self.out(h)
        return out

In [ ]:
timesteps = 200
base_ch = 96   

batch_size = 32
lr = 1e-4 

In [ ]:
# Construir vocabulario
all_labels = df["label"].tolist()
cleaned_labels = [re.sub(r"\d+", "", lbl).strip() for lbl in all_labels]
all_labels = list(set(cleaned_labels))
vocab = TextEmbedding.build_vocab(all_labels)

print(f" Vocabulario generado con {len(vocab)} palabras únicas:")
# print(vocab)

# Dataset y DataLoader
dataset = MinecraftDataset(df.to_dict("records"), image_size=img_size, augment=False)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


#### Entrenamiento

In [ ]:
# MODELOS 
txt_emb = TextEmbedding(vocab, embed_dim=embed_dim).to(device)
model = MiniUNet(embed_dim=embed_dim, base_ch=base_ch, timesteps=timesteps).to(device)

In [ ]:
optimizer = optim.Adam(list(model.parameters()) + list(txt_emb.parameters()), lr=lr)


In [ ]:
# entrenamiento funcion primera version
num_epochs = 8001
# Listas para guardar el historial
historial_epocas = []
historial_loss = []

for epoch in range( num_epochs):
    model.train()
    total_loss = 0

    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        cond = txt_emb(labels).to(device)

        # sample random timesteps y ruido
        t = torch.randint(0, timesteps, (imgs.size(0),), device=device)
        noise = torch.randn_like(imgs)

        alpha_bar_t = model.alphas_cumprod[t].view(-1, 1, 1, 1)
        x_t = torch.sqrt(alpha_bar_t) * imgs + torch.sqrt(1 - alpha_bar_t) * noise
        
        noise_pred = model(x_t, t, cond)
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # =========================================================
    # LOGGING
    # =========================================================   
    if epoch % 500 == 0:
        test_prompt = "iron boat"
        historial_epocas.append(epoch)
        historial_loss.append(total_loss / len(dataloader))

        print(f"Epoch [{epoch}/{num_epochs}] ")
        print(f" Loss Total: {total_loss / len(dataloader):.4f} ")

        mostrar_metricas(historial_epocas, historial_loss, num_epochs, "loss Total")
        
        # Generar una imagen de ejemplo condicional
        mostrar_resultados(generar_imagenes(model, txt_emb, test_prompt, n_samples=4))
        

#### Guardar modelos

In [ ]:
# GUARDAR MODELOS
import os
import torch

version = "vf1"
save_dir = f"./modelos/{version}"
os.makedirs(save_dir, exist_ok=True)


torch.save(model.state_dict(), os.path.join(save_dir, "diffusion_model.pth"))
torch.save(txt_emb.state_dict(), os.path.join(save_dir, "text_embedding.pth"))

# guardamos hiperparámetros y vocabulario
metadata = {
    "embed_dim": embed_dim,
    "timesteps": timesteps,
    "base_ch": base_ch,
    "vocab": txt_emb.vocab,}

torch.save(metadata, os.path.join(save_dir, "metadata.pth"))

print(f" Modelos y metadatos guardados en: {save_dir}")
print(f" Metadata info: {metadata}")


#### resultados

In [ ]:
prompt = "golden door"
imgs = generar_imagenes(model, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "emerald pickaxe"
imgs = generar_imagenes(model, txt_emb, prompt)
mostrar_resultados(imgs)


In [ ]:
prompt = "redstone ingot"
imgs = generar_imagenes(model, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "blue slime dye"
imgs = generar_imagenes(model, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "rotten flesh"
imgs = generar_imagenes(model, txt_emb, prompt)
mostrar_resultados(imgs)

#### Importar modelos

In [ ]:
import torch
import os

# Configuración
device = "cuda" if torch.cuda.is_available() else "cpu"
version = "vf1"
save_dir = f"./modelos/{version}"

metadata = torch.load(os.path.join(save_dir, "metadata.pth"))
txt_emb = TextEmbedding(vocab=metadata["vocab"], embed_dim=metadata["embed_dim"]).to(device)
model = MiniUNet(embed_dim=metadata["embed_dim"], base_ch=metadata["base_ch"], timesteps=metadata["timesteps"]).to(device)

# === Cargar pesos ===
model.load_state_dict(torch.load(os.path.join(save_dir, "diffusion_model.pth"), map_location=device))
txt_emb.load_state_dict(torch.load(os.path.join(save_dir, "text_embedding.pth"), map_location=device))

print(" Modelos cargados correctamente desde", save_dir)


#### Validación de métricas de resultados

In [ ]:
prompts_inventados = [
    "bamboo sword",
    "glass pickaxe",
    "magma boat",
    "slime boots",
    "copper apple",
    "phantom shield",
    "amethyst door",
    "netherite bone",
    "rotten cake",
    "spider bow",
    "wooden chestplate",
    "baked melon",
    "glowstone helmet",
    "snow sword",
    "crimson water",
    "iron bamboo",
    "diamond chicken",
    "emerald fish",
    "enchanted clay",
    "turtle axe",
    "golden paper",
    "lapis apple",
    "beef pie",
    "quartz boat",
    "redstone sword",
    "leather compass",
    "honeycomb shield",
    "chorus bread",
    "bone compass",
    "poisonous cake"
]

In [ ]:

import torch
import gc
import random
from transformers import CLIPProcessor, CLIPModel, BlipProcessor, BlipForImageTextRetrieval
import transformers
import random

def generar_prompts_aleatorios(txt_emb, n_samples=4, palabras_por_prompt=2):
    # Filtramos la palabra "unknown" para que no aparezca en las combinaciones
    vocab_limpio = [w for w in txt_emb.vocab if w != "unknown"]
    prompts_generados = [] 

    for _ in range(n_samples):
        # Elegimos palabras al azar con reemplazo
        palabras_elegidas = random.choices(vocab_limpio, k=palabras_por_prompt)
        prompt = " ".join(palabras_elegidas)
        prompts_generados.append(prompt)

    return prompts_generados


def evaluar_clip_blip(model, txt_emb, n=200, batch_size=64, img_size=img_size, device=device):
    # Cargamos los modelos de evaluacion
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # Utilizamos el modelo ITM que tiene los pesos correctos para calcular similitud
    blip_model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    
    clip_model.eval()
    blip_model.eval()
    
    clip_score_total = 0.0
    blip_score_total = 0.0
    conteo = 0
    
    
    # Iteramos directamente sobre la cantidad n
    for i in range(0, n, batch_size):
        actual_n = min(batch_size, n - i)
        
        # Generamos solo la cantidad necesaria para este lote
        lote_textos = prompts_inventados
        
        # Generamos las imagenes llamando a la funcion externa
        resultados_gen = generar_imagenes(
            model, 
            txt_emb, 
            prompt=lote_textos,
            device=device, 
            return_pil=True
        )
        
        # Extraemos solo las imagenes PIL
        imagenes_pil = [res[0] for res in resultados_gen]
        
        with torch.no_grad():
            # Calcular CLIP Score
            inputs_clip = clip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_clip = clip_model(**inputs_clip)
            image_embeds = outputs_clip.image_embeds
            text_embeds = outputs_clip.text_embeds

            image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

            clip_scores = (image_embeds * text_embeds).sum(dim=-1)

            clip_score_total += clip_scores.sum().item()
                    
            # Calcular BLIP Score
            inputs_blip = blip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_blip = blip_model(**inputs_blip)
            
            # Aplicamos softmax para obtener la probabilidad real de que el texto coincida con la imagen (indice 1)
            itm_probs = torch.nn.functional.softmax(outputs_blip.itm_score, dim=1)
            blip_score_total += itm_probs[:, 1].sum().item()
        
        conteo += actual_n
        
        # Limpieza de memoria
        torch.cuda.empty_cache()
        gc.collect()

    return {
        "CLIP_Score": clip_score_total / conteo,
        "BLIP_Score": blip_score_total / conteo
    }


for i in range(10):
    print(evaluar_clip_blip(model, txt_emb))

#### Tiempo de inferencia medio

In [ ]:
import time
import torch
import numpy as np

def medir_latencia_estadistica_texto(model, txt_emb, n_iterations=100,  device=device):
    """
    Calcula la media y desviación estándar del tiempo de inferencia para generar 
    una única imagen condicionada por un texto aleatorio distinto cada vez (Batch Size = 1).
    """
    model.eval()
    txt_emb.eval()
    tiempos = []
    
    print(f"Iniciando medición estadística sobre {n_iterations} iteraciones con prompts aleatorios...")

    with torch.no_grad():
        # Fase de calentamiento (Warm-up)
        # Generamos un prompt cualquiera para calentar la red
        prompt_warmup = generar_prompts_aleatorios(txt_emb, n_samples=1)
        
        for _ in range(5):
            _ = generar_imagenes(modelo=model, txt_emb=txt_emb, prompt=prompt_warmup, device=device, return_pil=False)
        
        if device == "cuda":
            torch.cuda.synchronize()

        # Ciclo de medición individual
        for i in range(n_iterations):
            # Generamos un nuevo prompt aleatorio exclusivo para esta iteración
            prompt_actual = generar_prompts_aleatorios(txt_emb, n_samples=1)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización previa
            
            t_inicio = time.perf_counter()
            
            # Llamada a la función de generación con el texto nuevo
            _ = generar_imagenes(modelo=model, txt_emb=txt_emb, prompt=prompt_actual, device=device, return_pil=False)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización posterior
            
            t_fin = time.perf_counter()
            tiempos.append(t_fin - t_inicio)

    # Cálculo de métricas descriptivas
    tiempos_arr = np.array(tiempos)
    mean_time = np.mean(tiempos_arr)
    std_time = np.std(tiempos_arr)
    
    print(f"Resultados de Inferencia (n=1 por iteración)")
    print(f"mean: {mean_time:.6f} s, std: {std_time:.6f} s")

    return {
        "mean": mean_time,
        "std": std_time
    }


medir_latencia_estadistica_texto(model, txt_emb)